<a href="https://colab.research.google.com/github/aayurchik/27_toxicity_prediction/blob/main/models/fingerptints_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Загрузка фингерпринтов и таргетов
fp_sparse = sparse.load_npz('/content/drive/MyDrive/Colab Notebooks/morgan_fp.npz')
fp_smiles = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fp_smiles.csv')
targets_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/2_targets_only.csv')
print(f"FP матрица: {fp_sparse.shape}, Таргеты: {targets_df.shape}")
# Объединение по SMILES
fp_dict = dict(zip(fp_smiles['smiles'], fp_smiles['index']))
targets_dict = {row['smiles']: row.drop('smiles').to_dict() for _, row in targets_df.iterrows()}
valid_indices = []
all_targets = []
for smiles, fp_idx in fp_dict.items():
    if smiles in targets_dict:
        valid_indices.append(fp_idx)
        all_targets.append(targets_dict[smiles])
X_fp = fp_sparse[valid_indices, :].toarray()
y_fp = pd.DataFrame(all_targets)
target_cols = y_fp.columns.tolist()
print(f"Объединено: X_fp={X_fp.shape}, y_fp={y_fp.shape}")

FP матрица: (339061, 2048), Таргеты: (339055, 14)
Объединено: X_fp=(339055, 2048), y_fp=(339055, 13)


In [6]:
# Функция обучения
def train_model(X, y_target, model):
    valid_idx = y_target.notna()
    X_valid = X[valid_idx]
    y_valid = y_target[valid_idx].astype(int)
    if len(np.unique(y_valid)) < 2:
        return None
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=42, stratify=y_valid)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = {
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)}
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        metrics['ROC-AUC'] = roc_auc_score(y_test, y_prob)
    return metrics
# Обучение моделей
results = []
for cat in target_cols:
    # KNN
    knn_metrics = train_model(X_fp, y_fp[cat], KNeighborsClassifier(n_neighbors=5))
    if knn_metrics:
        results.append({
            'tox_category': cat,
            'model': 'KNN',
            'F1': knn_metrics['F1'],
            'Precision': knn_metrics['Precision'],
            'Recall': knn_metrics['Recall'],
            'ROC-AUC': knn_metrics.get('ROC-AUC', None)})
    # Logistic Regression
    lr_metrics = train_model(X_fp, y_fp[cat], LogisticRegression(max_iter=1000, class_weight='balanced'))
    if lr_metrics:
        results.append({
            'tox_category': cat,
            'model': 'LogReg',
            'F1': lr_metrics['F1'],
            'Precision': lr_metrics['Precision'],
            'Recall': lr_metrics['Recall'],
            'ROC-AUC': lr_metrics.get('ROC-AUC', None)})
# Результаты
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['tox_category', 'model'])
print("\nитог")
print(f"{'Категория':<25} {'Модель':<8} {'F1':<8} {'Precision':<10} {'Recall':<10} {'ROC-AUC':<10}")
for _, row in results_df.iterrows():
    print(f"{row['tox_category']:<25} {row['model']:<8} "
          f"{row['F1']:.4f}    {row['Precision']:.4f}      {row['Recall']:.4f}      "
          f"{row['ROC-AUC'] if pd.notna(row['ROC-AUC']) else 'N/A':<10}")


итог
Категория                 Модель   F1       Precision  Recall     ROC-AUC   
acute_toxicity            KNN      0.4508    0.5668      0.3742      0.6938599135765249
acute_toxicity            LogReg   0.5424    0.5009      0.5914      0.7373912169631192
carcinogenicity           KNN      0.7170    0.6357      0.8221      0.5758095305429863
carcinogenicity           LogReg   0.6747    0.6763      0.6731      0.635004242081448
cardiotoxicity            KNN      0.5202    0.6957      0.4154      0.8520228200314266
cardiotoxicity            LogReg   0.3891    0.2574      0.7970      0.8867111227628038
dermal_toxicity           KNN      0.6913    0.7956      0.6111      0.656023967758294
dermal_toxicity           LogReg   0.7718    0.8224      0.7271      0.748011583526837
endocrine_metabolic_tox   KNN      0.3647    0.5830      0.2653      0.6700072554090004
endocrine_metabolic_tox   LogReg   0.5058    0.4833      0.5306      0.6752553345859765
genotoxicity              KNN      0.742

In [7]:
# ССравним результаты моделей по физхим признакам и фингерпринтам

physchem = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/results_physchem.csv')

print("LogReg сравнение:")
print(f"{'Категория':<25} {'PhysChem':<8} {'FP':<8} {'Δ':<6}")
for cat in target_cols:
    phys = physchem[(physchem['tox_category']==cat) & (physchem['model']=='LogReg')]['F1'].values[0]
    fp = results_df[(results_df['tox_category']==cat) & (results_df['model']=='LogReg')]['F1'].values[0]
    diff = fp - phys
    print(f"{cat:<25} {phys:.4f}  {fp:.4f}  {diff:+.4f}")

print("\nKNN сравнение:")
print(f"{'Категория':<25} {'PhysChem':<8} {'FP':<8} {'Δ':<6}")
for cat in target_cols:
    phys = physchem[(physchem['tox_category']==cat) & (physchem['model']=='KNN')]['F1'].values[0]
    fp = results_df[(results_df['tox_category']==cat) & (results_df['model']=='KNN')]['F1'].values[0]
    diff = fp - phys
    print(f"{cat:<25} {phys:.4f}  {fp:.4f}  {diff:+.4f}")

LogReg сравнение:
Категория                 PhysChem FP       Δ     
acute_toxicity            0.5208  0.5424  +0.0216
carcinogenicity           0.7533  0.6747  -0.0786
cardiotoxicity            0.2188  0.3891  +0.1703
dermal_toxicity           0.8352  0.7718  -0.0634
genotoxicity              0.6487  0.7537  +0.1050
hepatotoxicity            0.7531  0.7517  -0.0015
ocular_toxicity           0.9138  0.9015  -0.0123
oxidative_stress          0.2014  0.4562  +0.2548
respiratory_toxicity      0.8638  0.8627  -0.0011
neuro_sensory_toxicity    0.9119  0.8793  -0.0327
immuno_hematotoxicity     0.5279  0.5463  +0.0184
reprod_dev_toxicity       0.9783  0.9500  -0.0283
endocrine_metabolic_tox   0.4375  0.5058  +0.0683

KNN сравнение:
Категория                 PhysChem FP       Δ     
acute_toxicity            0.5577  0.4508  -0.1069
carcinogenicity           0.7021  0.7170  +0.0149
cardiotoxicity            0.5163  0.5202  +0.0039
dermal_toxicity           0.8210  0.6913  -0.1297
genotoxicity  

Ключевое наблюдение:

- LogReg лучше работает с фингерпринтами, а KNN хуже (страдает от проклятия размерности)
- Разные типы токсичности требуют разных признаков
- Физико-химические лучше в 7 из 13 категорий в logreg

Берем 2 типа признаков для каждой молекулы: физхим и фингерпринты.  
Обучаем 2 отдельные модели на одних и тех же данных и усредняем их показатели.  

Сравним два типа ансамблей:  
- LogReg-ансамбль    
- KNN-ансамбль  


In [9]:
# Загрузка физхим признаков
physchem_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/1_targets_and_features.csv')
physchem_feats = [col for col in physchem_df.columns if col.startswith('f')]
X_phys = physchem_df[physchem_feats].values[:len(X_fp)]  # матрица признаков
#  Ансамбль LOGREG
def ensemble_logreg(cat):
    """Ансамбль двух LogReg моделей"""
    y_cat = y_fp[cat].dropna().astype(int)
    idx = y_fp[cat].notna()
    if len(y_cat) < 100 or y_cat.nunique() < 2:
        return None
    # Разбиваем
    X_phys_train, X_phys_test, X_fp_train, X_fp_test, y_train, y_test = train_test_split(
        X_phys[idx], X_fp[idx], y_cat, test_size=0.2, random_state=42, stratify=y_cat)
    # LogReg на PhysChem
    lr_phys = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    lr_phys.fit(X_phys_train, y_train)
    proba_phys = lr_phys.predict_proba(X_phys_test)[:, 1]
    # LogReg на FP
    lr_fp = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    lr_fp.fit(X_fp_train, y_train)
    proba_fp = lr_fp.predict_proba(X_fp_test)[:, 1]
    # Усредняем
    proba_ensemble = (proba_phys + proba_fp) / 2
    y_pred = (proba_ensemble > 0.5).astype(int)
    return f1_score(y_test, y_pred)

# Ансамбль KNN

def ensemble_knn(cat):
    """Ансамбль двух KNN моделей"""
    y_cat = y_fp[cat].dropna().astype(int)
    idx = y_fp[cat].notna()
    if len(y_cat) < 100 or y_cat.nunique() < 2:
        return None
    # Разбиваем
    X_phys_train, X_phys_test, X_fp_train, X_fp_test, y_train, y_test = train_test_split(
        X_phys[idx], X_fp[idx], y_cat, test_size=0.2, random_state=42, stratify=y_cat)
    # KNN на PhysChem
    knn_phys = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn_phys.fit(X_phys_train, y_train)
    proba_phys = knn_phys.predict_proba(X_phys_test)[:, 1]
    # KNN на FP
    knn_fp = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn_fp.fit(X_fp_train, y_train)
    proba_fp = knn_fp.predict_proba(X_fp_test)[:, 1]
    # Усредняем
    proba_ensemble = (proba_phys + proba_fp) / 2
    y_pred = (proba_ensemble > 0.5).astype(int)
    return f1_score(y_test, y_pred)

print(f"{'Категория':<25} {'LogReg':<8} {'KNN':<8} {'Лучший ансамбль':<12}")
for cat in target_cols:
    f1_logreg = ensemble_logreg(cat)
    f1_knn = ensemble_knn(cat)
    if f1_logreg and f1_knn:
        best_ensemble = 'LogReg' if f1_logreg > f1_knn else 'KNN'
        print(f"{cat:<25} {f1_logreg:.4f}    {f1_knn:.4f}    {best_ensemble}")

Категория                 LogReg   KNN      Лучший ансамбль
----------------------------------------------------------------------
acute_toxicity            0.5644    0.3642    LogReg
carcinogenicity           0.7136    0.7549    KNN
cardiotoxicity            0.3943    0.2198    LogReg
dermal_toxicity           0.8066    0.7766    LogReg
genotoxicity              0.7671    0.6992    LogReg
hepatotoxicity            0.7772    0.7637    LogReg
ocular_toxicity           0.9019    0.9211    KNN
oxidative_stress          0.4562    0.0342    LogReg
respiratory_toxicity      0.8898    0.8941    KNN
neuro_sensory_toxicity    0.8848    0.8987    KNN
immuno_hematotoxicity     0.5785    0.3562    LogReg
reprod_dev_toxicity       0.9552    0.9808    KNN
endocrine_metabolic_tox   0.5320    0.3105    LogReg


**Результаты:**  
LogReg-ансамбль лучше в 9 категориях  
KNN-ансамбль лучше в 4 категориях  
Это улучшение по сравнению с использованием только одного типа признаков  

**Проверим гипотезу:**  
  
объединение физико-химических признаков с отобранными фингерпринтами в одной модели улучшает предсказательную способность по сравнению с использованием только одного типа признаков  

Отбираем 15% самых информативных фингерпринтов по p-value на репрезентативных категориях. Создаём единый вектор признаков и обучаем Logreg на комбинированных признаках.  


```python
# Селекция топ-15% фингерпринтов
selection_cats = ['genotoxicity', 'acute_toxicity', 'oxidative_stress']
pvalues_total = np.zeros(X_fp.shape[1])

for cat in selection_cats:
    y_cat = y_fp[cat].dropna().astype(int)
    idx = y_fp[cat].notna()
    if len(y_cat) > 100:
        _, p_values = f_classif(X_fp[idx], y_cat)
        pvalues_total += p_values

top_percent = 15
n_select = int(X_fp.shape[1] * top_percent / 100)
top_idx = np.argsort(-pvalues_total)[:n_select]
X_fp_sel = X_fp[:, top_idx]

# Комбинированные признаки
X_combined = np.hstack([X_phys, X_fp_sel])

# Обучение
def train_simple(X, y_target):
    valid_idx = y_target.notna()
    X_valid = X[valid_idx]
    y_valid = y_target[valid_idx].astype(int)
    if len(np.unique(y_valid)) < 2:
        return None
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=42, stratify=y_valid)
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    lr.fit(X_train, y_train)
    return f1_score(y_test, lr.predict(X_test))

# Результаты
print("Комбинированные принзнаки (LogReg)")
print(f"{'Категория':<25} {'F1':<8}")
for cat in target_cols:
    f1 = train_simple(X_combined, y_fp[cat])
    if f1:
        print(f"{cat:<25} {f1:.4f}")


| Категория                 | F1      |
|---------------------------|---------|
| acute_toxicity            | 0.4748  |
| carcinogenicity           | 0.6505  |
| cardiotoxicity            | 0.2159  |
| dermal_toxicity           | 0.7329  |
| genotoxicity              | 0.6206  |
| hepatotoxicity            | 0.7137  |
| ocular_toxicity           | 0.8449  |
| oxidative_stress          | 0.2333  |
| respiratory_toxicity      | 0.8286  |
| neuro_sensory_toxicity    | 0.8000  |
| immuno_hematotoxicity     | 0.4958  |
| reprod_dev_toxicity       | 0.9053  |
| endocrine_metabolic_tox   | 0.4944  |

Комбинирование признаков в одной модели оказалось неэффективным.
физико-химические и фингерпринты лучше работают отдельно.
Используем ансамбли, а не объединение в одной модели.

Дальнейшая работа:

- используем более мощные модели для эмбедингов
- разные способы ансамблирования признаков
- посмотрим что лучше работает на разных категориях токсичности
- поработаем с гиперпараметрами

**Подберем гиперпараметры** (в процессе)


In [ ]:
# # Тюнинг параметров ансамблей
# # Функция для ансамбля с параметрами
# def ensemble_logreg_cat(cat, C_value=1.0):
#     """Ансамбль LogReg с настраиваемыми параметрами"""
#     y_cat = y_fp[cat].dropna().astype(int)
#     idx = y_fp[cat].notna()
#     if len(y_cat) < 100 or y_cat.nunique() < 2:
#         return None
#     # Разбиваем данные
#     X_phys_train, X_phys_test, X_fp_train, X_fp_test, y_train, y_test = train_test_split(
#         X_phys[idx], X_fp[idx], y_cat, test_size=0.2, random_state=42, stratify=y_cat)
#     # LogReg на PhysChem
#     lr_phys = LogisticRegression(
#         C=C_value,
#         max_iter=500,
#         class_weight='balanced',
#         solver='saga',
#         n_jobs=1)
#     lr_phys.fit(X_phys_train, y_train)
#     # LogReg на FP
#     lr_fp = LogisticRegression(
#         C=C_value,
#         max_iter=500,
#         class_weight='balanced',
#         solver='saga',
#         n_jobs=1)
#     lr_fp.fit(X_fp_train, y_train)
#     # Усредняем вероятности
#     proba_phys = lr_phys.predict_proba(X_phys_test)[:, 1]
#     proba_fp = lr_fp.predict_proba(X_fp_test)[:, 1]
#     proba_ensemble = (proba_phys + proba_fp) / 2
#     y_pred = (proba_ensemble > 0.5).astype(int)
#     return f1_score(y_test, y_pred)
# # Пробные параметры C для тюнинга
# C_values = [0.01, 0.1, 1.0, 10.0]
# # Тюнинг на 2 проблемных категориях
# tuning_cats = ['oxidative_stress', 'cardiotoxicity']
# for cat in tuning_cats:
#     print(f"\nТюнинг ансамбля: {cat}")
#     # Baseline (C=1.0)
#     baseline_f1 = ensemble_logreg_cat(cat, C_value=1.0)
#     if baseline_f1 is None:
#         continue
#     print(f"Baseline (C=1.0): {baseline_f1:.4f}")
#     # Перебираем разные значения C
#     best_f1 = baseline_f1
#     best_C = 1.0
#     for C_val in C_values:
#         f1 = ensemble_logreg_cat(cat, C_value=C_val)
#         if f1 is None:
#             continue
#         print(f"  C={C_val:.2f}: {f1:.4f}", end="")
#         if f1 > best_f1:
#             best_f1 = f1
#             best_C = C_val
#             print(f" новый лучший")
#         else:
#             print("")
#     print(f"Лучший C: {best_C} (F1={best_f1:.4f})")
#     if best_f1 > baseline_f1:
#         print(f"Улучшение: +{best_f1 - baseline_f1:.4f}")
#     else:
#         print(f"Без изменений")

# # тюнинг весов ансамбля

# def ensemble_weighted(cat, weight_phys=0.5):
#     """Ансамбль с разными весами моделей"""
#     y_cat = y_fp[cat].dropna().astype(int)
#     idx = y_fp[cat].notna()
#     if len(y_cat) < 100 or y_cat.nunique() < 2:
#         return None
#     X_phys_train, X_phys_test, X_fp_train, X_fp_test, y_train, y_test = train_test_split(
#         X_phys[idx], X_fp[idx], y_cat, test_size=0.2, random_state=42, stratify=y_cat)
#     # Обучаем модели
#     lr_phys = LogisticRegression(max_iter=500, class_weight='balanced', solver='saga', n_jobs=1)
#     lr_phys.fit(X_phys_train, y_train)
#     lr_fp = LogisticRegression(max_iter=500, class_weight='balanced', solver='saga', n_jobs=1)
#     lr_fp.fit(X_fp_train, y_train)
#     # Взвешенное усреднение
#     proba_phys = lr_phys.predict_proba(X_phys_test)[:, 1]
#     proba_fp = lr_fp.predict_proba(X_fp_test)[:, 1]
#     # Веса: PhysChem * weight_phys + FP * (1 - weight_phys)
#     proba_ensemble = weight_phys * proba_phys + (1 - weight_phys) * proba_fp
#     y_pred = (proba_ensemble > 0.5).astype(int)
#     return f1_score(y_test, y_pred)

# # Тестируем разные веса
# weights = [0.0, 0.25, 0.5, 0.75, 1.0]  # 0.0 = только FP, 1.0 = только PhysChem
# for cat in ['oxidative_stress', 'cardiotoxicity', 'acute_toxicity']:
#     print(f"\nВеса ансамбля: {cat}")
#     baseline_weight = 0.5  # равные веса
#     baseline_f1 = ensemble_weighted(cat, weight_phys=baseline_weight)
#     print(f"Baseline (вес 0.5): {baseline_f1:.4f}")
#     best_f1 = baseline_f1
#     best_weight = 0.5
#     for w in weights:
#         f1 = ensemble_weighted(cat, weight_phys=w)
#         if f1 is None:
#             continue
#         print(f"Вес PhysChem={w}: {f1:.4f}", end="")
#         if f1 > best_f1:
#             best_f1 = f1
#             best_weight = w
#             print(f"  лучше")
#         else:
#             print("")
#     print(f"Оптимальный вес PhysChem: {best_weight} (F1={best_f1:.4f})")